/Users/hyin/usgs_mendenhall/events/2026-04-14_nevada/wisp_inversion/mask_s2_observations.py

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, box
import os
from pathlib import Path


insar_data = "/Volumes/T9_InSAR/2026-10-09_panama/gcent/products/nisar-azoff-p076-20261005-20261017_resampled.txt"
parent_dir = str(Path(insar_data).parent)
file_stem = Path(insar_data).stem

mask_files = [
    "/Users/hyin/usgs_mendenhall/events/20261009_panama/simple-fault_west.shp",       # line or polygon, shp or geojson
    # "/Users/hyin/usgs_mendenhall/events/20261009_panama/map_layers/mask_layer/A004_scene-edge.geojson",
]
# mask_files = []   # or None, for no masking
# mask_file = None
buffer_distance = 1000        # meters

insar_out = f"{parent_dir}/{file_stem}_{buffer_distance}m.txt"

crop = False
bl = [7.1890,-81.0233]  # lat, lon (copied from QGIS)
tr = [7.6339,-80.2402] 

# Define a zero point to shift all disp values to be centered around zero
zero_level = 36
# zero_level = '/Volumes/T9_InSAR/2026-10-09_panama/gcent/products/zero_point.txt'

# ---------------------------------------------------------
# Read mask geometry
# ---------------------------------------------------------
geometry = None

if mask_files:
    parts = []
    for mask_file in mask_files:
        # Keep everything in geographic coordinates (lon/lat)
        mask = gpd.read_file(mask_file).to_crs("EPSG:4326")
        # Merge all features of this file into one geometry
        geom = mask.union_all()

        # Buffer only if it is a line. buffer_distance is in meters, so
        # briefly project to a local metric CRS (estimated from the data)
        # for the buffer, then convert the result back to lon/lat.
        if geom.geom_type in ["LineString", "MultiLineString"]:
            metric_crs = mask.estimate_utm_crs()
            geom = (
                gpd.GeoSeries([geom], crs="EPSG:4326")
                .to_crs(metric_crs)
                .buffer(buffer_distance)
                .to_crs("EPSG:4326")
                .iloc[0]
            )
        parts.append(geom)

    # Combine all (buffered) mask geometries into one
    geometry = gpd.GeoSeries(parts, crs="EPSG:4326").union_all()

# ---------------------------------------------------------
# Apply bounding box crop if requested
# ---------------------------------------------------------


if crop == True:
    # Build the bounding box directly in lon/lat from the corners.
    # bl/tr are [lat, lon], so lon is index 1 and lat is index 0.
    minx, maxx = sorted([bl[1], tr[1]])
    miny, maxy = sorted([bl[0], tr[0]])
    bbox_geom = box(minx, miny, maxx, maxy)

    print(f"box bounds (minx,miny,maxx,maxy): {bbox_geom.bounds}")
else:
    bbox_geom = None

# NOTE: the mask geometry (`geometry`) and the crop box (`bbox_geom`) have
# OPPOSITE senses and are kept separate (not intersected):
#   * bbox  -> KEEP points INSIDE the box
#   * mask  -> EXCLUDE points INSIDE the (buffered) mask
# A point is kept only if it is inside the bbox AND outside the mask-buffer.

# ---------------------------------------------------------
# Prepare zero-level shift
# ---------------------------------------------------------
# All displacement values will have zero_level subtracted so the
# field is centered around zero. Set zero_level to None to skip.
if zero_level is not None:
    zero_shift = float(zero_level)
else:
    zero_shift = 0


# ---------------------------------------------------------
# Filter InSAR points
# ---------------------------------------------------------

with open(insar_out, "w") as out:

    with open(insar_data) as infile:

        for line in infile:

            if line.startswith("#"):
                out.write(line)
                continue

            lon, lat, displacement, sx, sy, sz = line.split()

            # Point in geographic coordinates (lon/lat)
            point = Point(float(lon), float(lat))

            # Combined filter:
            #   keep if inside the crop box (if any) ...
            inside_bbox = bbox_geom is None or bbox_geom.contains(point)
            #   ... AND outside the (buffered) mask (if any).
            outside_mask = geometry is None or not geometry.contains(point)

            if inside_bbox and outside_mask:
                # Subtract the zero level from the displacement value
                displacement = float(displacement) - zero_shift
                out.write(
                    f"{lon} {lat} {displacement} {sx} {sy} {sz}\n"
                )